# GPT-SoVITS fine-tune — cool-jahns (HARDENED / DEBUGGED)

**This is the debugged notebook** — every failure hit during the first run is fixed here so the redo is one-shot:
- All **11 missing deps** baked into the install cell (no more crash-fix loop): `ffmpeg-python`, `wordsegment`, `g2p_en` (+NLTK data), `x_transformers`, `pytorch_lightning`, `fast_langdetect`, `split_lang`, `cn2an`, `pypinyin`, `jieba`, `jieba_fast`.
- Data loaded **directly from mounted Drive** (no gdown / no share-link needed).
- **AUTO-SAVE weights to Drive** cell (cell 7) — run it the moment training finishes so the weights survive the VM recycling. THIS is the fix for losing the first run.
- Colab-specific settings noted inline: slicer **CPU threads = 2**, training **batch size = 4** (T4-safe).

**Run on a GPU runtime** (Runtime → Change runtime type → T4/L4 GPU). On free tier a GPU may be refused — Colab Pay-As-You-Go (~$10 / 100 units) guarantees one.

> Training happens in the GPT-SoVITS **WebUI** (cell 6 prints a gradio link). After training, **come back and run cell 7 to save the weights to Drive.**

In [ ]:
# 1. Confirm a GPU is attached (if this errors, set Runtime -> Change runtime type -> GPU)
!nvidia-smi

In [ ]:
# 2. Clone + install EVERYTHING (all deps that crashed the first run are here)
%cd /content
!git clone https://github.com/RVC-Boss/GPT-SoVITS
%cd /content/GPT-SoVITS
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q -r requirements.txt
# ASR + slicer deps used by the preprocessing tabs
!pip install -q faster-whisper funasr modelscope
# --- deps discovered missing during first-run debugging (bake in so no crash-loop) ---
!pip install -q ffmpeg-python wordsegment g2p_en x_transformers pytorch_lightning \
    fast_langdetect split_lang cn2an pypinyin jieba jieba_fast
# NLTK data that g2p_en needs (English G2P) — not auto-downloaded
import nltk
for d in ['averaged_perceptron_tagger_eng','averaged_perceptron_tagger','cmudict','punkt','punkt_tab']:
    nltk.download(d)
print('install done')

In [ ]:
# 3. Pretrained models (~5 GB). Newer GPT-SoVITS also auto-fetches on first use.
from huggingface_hub import snapshot_download
snapshot_download('lj1995/GPT-SoVITS', local_dir='GPT_SoVITS/pretrained_models')
print('pretrained models staged')

In [ ]:
# 4. Mount Drive (for BOTH loading data and SAVING weights). Click through the OAuth prompt.
from google.colab import drive
drive.mount('/content/drive')
import os
print('mounted:', os.path.ismount('/content/drive'))

In [ ]:
# 5. Load the training data straight from Drive (no gdown / no share-link).
#    ADJUST DATA_ZIP if your folder name differs in this account's Drive.
import os
DATA_ZIP = '/content/drive/MyDrive/finetune-data-repo/TTS/jahns-full-24k.zip'
assert os.path.exists(DATA_ZIP), f'NOT FOUND: {DATA_ZIP}\nCheck the folder name in THIS account\'s Drive and fix the path.'
!mkdir -p /content/jahns
!unzip -o -q "$DATA_ZIP" -d /content/jahns
print('files:'); import glob; print([os.path.basename(f) for f in glob.glob('/content/jahns/*.wav')])

In [ ]:
# 6. Launch the GPT-SoVITS WebUI (prints a public gradio link). Do the steps below in it.
import os
os.environ['is_share'] = 'True'
!python webui.py

## In the WebUI (open the gradio link)

**Tab `0-Fetch datasets` → 0-Preprocessing:**
1. **Audio slicing** — input dir `/content/jahns`. ⚠️ Set **CPU threads = 2** (Colab has 2 vCPUs; the default 4 errors). Run.
2. **ASR** — model **Faster Whisper**, language **en**. Run → produces the `.list` label file (~427 segments from 59 min).

**Tab `1-GPT-SoVITS-TTS` → 1A-Dataset Formatting:**
3. Experiment name `cool-jahns`. Run **One-click formatting** (1Aa/1Ab/1Ac).

**→ 1B-Fine-Tuning:**
4. **Train SoVITS** — set **batch size = 4** (T4-safe), 8 epochs. ~10-15 min.
5. **Train GPT** — set **batch size = 4**, 15 epochs. ~5-10 min.

**⚠️ THE MOMENT BOTH FINISH: come back here and run cell 7 below to save the weights to Drive.** Do NOT close the tab or let it idle first — that is how the first run was lost.

**→ 1C-Inference** (to hear it): click **refreshing model paths**, pick `cool-jahns-e15.ckpt` + `cool-jahns_e8_s904.pth`, **Open TTS Inference WebUI**. In it: check **Enable no reference mode**, set language **English**, upload any 3-10 s Jahns clip as the reference audio (needed for timbre even in no-ref mode), type text, synthesize.

In [ ]:
# 7. *** SAVE WEIGHTS TO DRIVE — RUN IMMEDIATELY AFTER TRAINING FINISHES ***
# This is the fix for losing the first run. Weights live only on the ephemeral VM until copied here.
from google.colab import drive
import os, glob, shutil
drive.mount('/content/drive')  # no-op if already mounted
SAVE_DIR = '/content/drive/MyDrive/finetune-data-repo/TTS/cool-jahns-weights'
os.makedirs(SAVE_DIR, exist_ok=True)
found = []
for pat in ['/content/GPT-SoVITS/SoVITS_weights*/*cool-jahns*.pth',
            '/content/GPT-SoVITS/GPT_weights*/*cool-jahns*.ckpt']:
    for f in glob.glob(pat):
        shutil.copy(f, SAVE_DIR)
        found.append(os.path.basename(f))
        print('saved ->', os.path.basename(f), f'({os.path.getsize(f)//1024//1024} MB)')
assert found, 'NO WEIGHTS FOUND — did training finish? check SoVITS_weights*/ and GPT_weights*/'
print('\nWEIGHTS SAFE IN DRIVE:', SAVE_DIR)
print('contents:', os.listdir(SAVE_DIR))

## After cell 7

Your two weight files are now in `finetune-data-repo/TTS/cool-jahns-weights/` in Drive — **permanent, downloadable, survive the VM.** You need BOTH to synthesize:
- `cool-jahns_e8_s904.pth` (SoVITS — timbre)
- `cool-jahns-e15.ckpt` (GPT — prosody)

To reuse later: put both back under `SoVITS_weights_v2Pro/` and `GPT_weights_v2Pro/`, launch the WebUI, load them in 1C-Inference.